# 03 — Deep models on a GPU (option C) — Colab or Kaggle

Trains **N-HiTS** and **TFT** (neuralforecast) for each backtest origin and writes their forecasts
next to the inputs, so they can be scored against LightGBM with `idf-deep score`.

Inputs come from `idf-deep export` (`data/deep_colab/inputs/`): one file per origin with the
history (target `log1p(entries)`, `available_mask` = 0 on excluded days) and the future
exogenous variables, already built with the backtest's no-leakage rules. This notebook only
trains and predicts.

**Variants** (`VARIANT`, first code cell after the install): `navigo_deep` = the option C
backtest (data up to the day before each origin, h = 60); `navigo_deep_step4` = the
production-like backtest of step 4 (`idf-deep export --warmup --lag 120 --weather
production`): history ends at the data cutoff (origin − 120 days), the model forecasts
the gap and the 60 scored days (h = 180, from `meta.json`); weather after day 16 is
climatology. Each variant has its own folder and zip in *My Drive*, so outputs never mix.

**Steps**
1. Put the folder `VARIANT` (with `inputs/` inside) in *My Drive*, or just the file
   `VARIANT.zip` at the top of *My Drive* (it is unpacked on the first run).
2. Runtime → Change runtime type → **L4 GPU** (T4 works; A100 is faster for TFT). Not TPU.
3. Runtime → Run all. Each model × origin is saved as soon as it is done; after a disconnect,
   run all again: finished ones are skipped.
4. Download `My Drive/VARIANT/outputs.zip` (made by the last cell) and unzip it into
   `idf-crowding/data/deep_colab/outputs/` (`navigo_deep`) or
   `idf-crowding/data/deep_colab/lag120_prod/outputs/` (`navigo_deep_step4`), then run
   `make deep-score` / `uv run idf-deep score --lag 120 --weather production`.

**On Kaggle** (when Colab has no GPU): the notebook detects `/kaggle/input` and needs no
Drive.
1. kaggle.com → Datasets → New dataset → upload `VARIANT.zip` (Kaggle unpacks it) → Create.
2. New notebook → File → Import notebook → this file. Settings (right panel): Accelerator =
   **GPU T4 x2** (the P100 may not work with recent PyTorch), Internet = On (needs a
   verified phone number). Add input → the dataset of step 1.
3. Set `PART` (one entry of `RUNS` per run), then **Save Version → Save & Run All (Commit)**:
   it runs in the background (up to 12 h) and keeps `/kaggle/working`. The loop stops
   starting new fits after `TIME_BUDGET_H` hours so the version saves before Kaggle's limit.
4. Unfinished part: Add input → this notebook's own output (Notebook Output Files), commit
   again: files already in an attached `outputs/` folder are reused, not recomputed.
5. Download `outputs.zip` from each version's Output tab and unzip them all into the same
   folder on the Mac. GPU quota: ~30 h per week.

`navigo_deep_production` = the production forecast (`idf-deep export --production`): one
origin (the data cutoff), h = 225, RUNS = TFT seeds 1–3 automatically; then
`idf-forecast combine --deep <outputs folder>`.

**Runs** (Settings cell): each run = model × seed × set of origins (`scored` = the 12
backtest origins, `warmup` = the 12 origins before them, which only feed the forecast log
used to calibrate the ensemble's intervals). Seed 1 keeps the plain model name (`tft`), so
files from earlier runs are reused; other seeds are saved as `tft_s2`, `tft_s3`, …

To split the runs across parallel Colab sessions (Colab Pro), set `PART` = 1, 2, … in the
Settings cell of each session (one entry of `RUNS` each).

A newer `VARIANT.zip` at the top of *My Drive* is unpacked automatically (inputs only;
`outputs/` and this notebook are never overwritten).

Expected time on an L4 at h = 60: TFT ~5–15 min per origin, N-HiTS ~2–5 min. At h = 180
(step 4) a fit is slower and needs more GPU memory: on an out-of-memory error the fit is
retried with half the `windows_batch_size` (recorded in the output column of that name).
Validation (early stopping) = the last h days before the data cutoff (neuralforecast needs
`val_size` ≥ h), so with h = 180 the weights are fitted on windows ending 180 days before
the cutoff; the most recent history still enters through the 182-day input window.

In [ ]:
!nvidia-smi -L || echo "No GPU: Runtime > Change runtime type > L4 GPU"
# 3.3.0 keeps Colab's pandas 2.2.3 (3.3.1 pulls pandas 3, which breaks google-colab)
%pip install -q neuralforecast==3.3.0 pandas==2.2.3

In [ ]:
import json
import shutil
import time
import traceback
import zipfile
from pathlib import Path

# navigo_deep = option C runs (h = 60); navigo_deep_step4 = lag 120 + production weather
VARIANT = "navigo_deep_step4"
KAGGLE = Path("/kaggle/input").exists()
if KAGGLE:
    # inputs = a Kaggle dataset made from VARIANT.zip (unpacked by Kaggle), added as input
    every = sorted(Path("/kaggle/input").glob("**/inputs/meta.json"))
    hits = [h for h in every if h.parent.parent.name == VARIANT]
    hits = hits or (every if len(every) == 1 else [])  # a single attached dataset: use it
    assert hits, f"Add input -> the dataset made from {VARIANT}.zip (found: {every})"
    INP, BASE = hits[0].parent, Path("/kaggle/working")
    ZIP = None
else:
    from google.colab import drive

    drive.mount("/content/drive")
    DRIVE = Path("/content/drive/MyDrive")
    BASE = DRIVE / VARIANT
    INP = BASE / "inputs"
    ZIP = DRIVE / f"{VARIANT}.zip"
META_PATH = INP / "meta.json"
if ZIP is not None and ZIP.exists():
    with zipfile.ZipFile(ZIP) as z:
        new_meta = json.loads(z.read(f"{VARIANT}/inputs/meta.json"))
        old_meta = json.loads(META_PATH.read_text()) if META_PATH.exists() else None
        if new_meta != old_meta:  # unpack inputs only: outputs/ and the notebook stay as they are
            members = [m for m in z.namelist() if m.startswith(f"{VARIANT}/inputs/")]
            z.extractall(DRIVE, members=members)
            print(f"unpacked {len(members)} input files from {ZIP.name}")
assert META_PATH.exists(), f"inputs not found under {BASE}"
print(sorted(p.name for p in INP.glob("*.parquet") if "_static" not in p.name))

In [ ]:
# Settings. One run = (model, seed, origins); origins = "scored" or "warmup".
# Seed 1 keeps the plain name ("tft"): files from earlier runs are reused, not recomputed.
# Step 4 (navigo_deep_step4): production model = mean(lgbm, mean of 3 TFT seeds); the
# warm-up origins of every seed feed the forecast log that calibrates its 80 % band.
RUNS = [
    ("tft", 1, "scored"),
    ("tft", 1, "warmup"),
    ("tft", 2, "scored"),
    ("tft", 2, "warmup"),
    ("tft", 3, "scored"),
    ("tft", 3, "warmup"),
]
# Production inputs (`idf-deep export --production`, meta.json "production": true) replace
# RUNS by the 3 seeds on the single production origin (next cell).
# Parallel sessions: PART = 1 .. len(RUNS) runs only RUNS[PART - 1]; None runs all of them.
# Start part 1 first and wait for "unpacked ..." (or the file list) before starting the others.
PART = None
QUICK_TEST = False  # True: 12 stations, 6 steps, to check the setup in a few minutes
# Kaggle stops a session after 12 h: no new fit starts after this many hours (None = no limit)
TIME_BUDGET_H = 10.5 if KAGGLE else None

In [ ]:
import numpy as np
import pandas as pd
import torch
from neuralforecast import NeuralForecast
from neuralforecast.losses.pytorch import MQLoss
from neuralforecast.models import NHITS, TFT

OUT = BASE / ("outputs_test" if QUICK_TEST else "outputs")
OUT.mkdir(parents=True, exist_ok=True)
META = json.loads((INP / "meta.json").read_text())
H, FUTR, STAT, QS = META["horizon"], META["futr_exog"], META["stat_exog"], META["quantiles"]
ORIGINS = {
    role: [o["origin"] for o in META["origins"] if o.get("role", "scored") == role]
    for role in ("warmup", "scored", "production")
}
if META.get("production"):  # production forecast: 3 seeds, one origin (the data cutoff)
    RUNS = [("tft", s, "production") for s in (1, 2, 3)]
print({k: len(v) for k, v in ORIGINS.items()})
GPU = torch.cuda.is_available()
torch.set_float32_matmul_precision("high")

CONFIGS = {
    "nhits": dict(
        cls=NHITS,
        input_size=182,
        max_steps=3000,
        learning_rate=1e-3,
        batch_size=128,
        windows_batch_size=1024,
        scaler_type="robust",
        early_stop_patience_steps=5,
        val_check_steps=100,
        random_seed=1,
    ),
    "tft": dict(
        cls=TFT,
        input_size=182,
        hidden_size=64,
        n_head=4,
        max_steps=2000,
        learning_rate=1e-3,
        batch_size=64,
        windows_batch_size=128,
        scaler_type="robust",
        early_stop_patience_steps=5,
        val_check_steps=100,
        random_seed=1,
    ),
}


def run_name(model, seed):
    return model if seed == 1 else f"{model}_s{seed}"


def build(model, seed, windows_batch_size=None):
    c = dict(CONFIGS[model])
    cls = c.pop("cls")
    c["random_seed"] = seed
    if windows_batch_size:
        c["windows_batch_size"] = windows_batch_size
    if QUICK_TEST:
        c.update(max_steps=6, val_check_steps=3)
    return cls(
        h=H,
        futr_exog_list=FUTR,
        stat_exog_list=STAT,
        loss=MQLoss(quantiles=QS),
        valid_loss=MQLoss(quantiles=QS),
        alias=run_name(model, seed),
        accelerator="gpu" if GPU else "cpu",
        devices=1,
        enable_progress_bar=False,
        enable_model_summary=False,
        logger=False,
        **c,
    )


def pick(columns, name, q):
    """Column of quantile q in the MQLoss output (NAME-lo-80.0 / NAME-median / NAME-hi-80.0)."""
    pref = f"{name}-median" if q == 0.5 else f"{name}-lo-" if q < 0.5 else f"{name}-hi-"
    hits = [c for c in columns if c.startswith(pref)]
    if len(hits) != 1:
        raise KeyError(f"no unique column for {name} q={q} in {list(columns)}")
    return hits[0]


def run_origin(model, seed, origin):
    name = run_name(model, seed)
    out = OUT / f"{name}_{origin}.parquet"
    if out.exists():
        print(f"{name} {origin}: done already")
        return
    t0 = time.time()
    df = pd.read_parquet(INP / f"{origin}.parquet")
    static = pd.read_parquet(INP / f"{origin}_static.parquet")
    if QUICK_TEST:
        ids = static["unique_id"].iloc[:12]
        df, static = df[df.unique_id.isin(ids)], static[static.unique_id.isin(ids)]
    df["ds"] = pd.to_datetime(df["ds"])
    hist = df[~df.is_future][["unique_id", "ds", "y", "available_mask", *FUTR]]
    futr = df[df.is_future][["unique_id", "ds", *FUTR]]
    wbs = CONFIGS[model]["windows_batch_size"]
    while True:  # on a GPU out-of-memory error, retry with half the windows per batch
        try:
            nf = NeuralForecast(models=[build(model, seed, wbs)], freq="D")
            nf.fit(df=hist, static_df=static, val_size=H)  # validation = last H days
            fc = nf.predict(futr_df=futr)
            break
        except torch.OutOfMemoryError:
            if wbs <= 16:
                raise
            wbs //= 2
            torch.cuda.empty_cache() if GPU else None
            print(f"{name} {origin}: out of GPU memory, retrying with windows_batch_size={wbs}")
    fc = fc if "unique_id" in fc.columns else fc.reset_index()
    res = pd.DataFrame(
        {
            "station_id": fc["unique_id"].astype("int64"),
            "date": pd.to_datetime(fc["ds"]).dt.date,
            "origin": pd.Timestamp(origin).date(),
            "model": name,
            "windows_batch_size": wbs,
        }
    )
    for q in QS:  # back from log1p to entries
        res[f"q{int(round(q * 100))}"] = np.expm1(fc[pick(fc.columns, name, q)].to_numpy()).clip(
            min=0
        )
    assert len(res) == len(static) * H, (len(res), len(static))
    res.to_parquet(out, index=False)
    print(f"{name} {origin}: {len(static)} series, {time.time() - t0:.0f} s")

In [ ]:
THIS_RUNS = RUNS if PART is None else [RUNS[PART - 1]]
print("this session:", THIS_RUNS)
if KAGGLE:  # outputs of earlier versions (Add input -> Notebook Output Files) are reused
    want = {f"{run_name(m, s)}_{o}.parquet" for m, s, r in RUNS for o in ORIGINS[r]}
    for f in Path("/kaggle/input").glob(f"**/{OUT.name}/*.parquet"):
        if f.name in want and not (OUT / f.name).exists():
            shutil.copy(f, OUT / f.name)
            print(f"reused {f.name}")
T_START = time.time()
todo = [(m, s, o) for m, s, r in THIS_RUNS for o in ORIGINS[r]]
for model, seed, origin in todo:
    if TIME_BUDGET_H and time.time() - T_START > TIME_BUDGET_H * 3600:
        print(f"time budget of {TIME_BUDGET_H} h reached: run again with this output attached")
        break
    try:
        run_origin(model, seed, origin)
    except Exception:
        print(f"{run_name(model, seed)} {origin}: FAILED")
        traceback.print_exc()
    finally:
        torch.cuda.empty_cache() if GPU else None

In [ ]:
done = sorted(p.name for p in OUT.glob("*.parquet"))
expected = [f"{run_name(m, s)}_{o}.parquet" for m, s, r in RUNS for o in ORIGINS[r]]
missing = [f for f in expected if f not in done]
mine = [f"{run_name(m, s)}_{o}.parquet" for m, s, r in THIS_RUNS for o in ORIGINS[r]]
print(f"this session: {'done' if not set(mine) & set(missing) else 'MISSING files'}")
print(f"{len(done)} files in outputs; missing for RUNS: {missing or 'none'}")
archive = shutil.make_archive(str(BASE / OUT.name), "zip", OUT)
print("download:", archive)